In [ ]:
# Same pinned set as s05_oot / s06b (the project has no requirements.lock yet)
%pip install optbinning "numpy<2" "pyarrow==14.0.1" "statsmodels>=0.14.4" lightgbm interpret-core "scipy>=1.12,<1.14" openpyxl --quiet
dbutils.library.restartPython()

In [ ]:
# s05c: discriminatory power of the locked Model C (FTS v1.0.1 and no-FTS v1.0.1-nofts) against the current decision score
# (A_SCORE) and the bureau score (AECB_SCORE, reference only) on OOT GOOD/BAD (spec 8.1, 8.3). Model A of both variants as reference.
# REPORT ONLY. Reads the locked registries, acard_master and the benchmark tables (read-only); writes only under
# <config.mdd.doc_output_root>/reports/benchmark/. Loading, E13 scoring and benchmark joins: benchmark_utils (same code as
# s06b_cutoff_analysis). AUC / Gini / KS with stratified bootstrap CIs and paired deltas: common (same functions as E13).
# Primary view = COMMON sample (OOT GOOD/BAD with all four scores non-missing); secondary view = each score on its own sample.
import sys, os, json, time, shutil, tempfile, importlib, warnings, hashlib
from datetime import datetime
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
importlib.reload(common)
import benchmark_utils as bu
importlib.reload(bu)
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", message="Missing values detected")

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
SEED = cfg["random_seed"]
np.random.seed(SEED)
CA, MDD = cfg["cutoff_analysis"], cfg["mdd"]
MODELS_USED = list(CA["models"])
assert MODELS_USED == ["A", "C"], f"this module scores Models A and C only, config has {MODELS_USED}"
LABELLED = list(cfg["labelled_statuses"])
assert set(LABELLED) == {"GOOD", "BAD"}, f"labelled_statuses must be GOOD + BAD, got {LABELLED}"
BASE_STATUSES = list(CA["approval_base_statuses"])
BOOT = cfg["selection"]["bootstrap"]
SPLITS = cfg["oot_report_splits"]
PERIODS = ["ALL"] + list(SPLITS)
TOL = float(cfg["deployment"]["numeric_tol"])
BENCHES, BENCH_KEYS, BENCH_NAME_OF = bu.parse_benchmarks(CA, MODELS_USED)
assert set(BENCH_KEYS) == {"A_SCORE", "AECB_SCORE"}, f"expected benchmarks A_SCORE and AECB_SCORE, got {BENCH_KEYS}"
FTS_CFG = CA.get("compare_with_fts") or {}
assert FTS_CFG.get("enabled") and list(FTS_CFG["models"]) == MODELS_USED, "cutoff_analysis.compare_with_fts must be enabled for A, C"
FTS_REG_PATH = FTS_CFG["registry"]

ARTIFACTS = common.artifacts_path(cfg)
REG_PATH = os.path.join(ARTIFACTS, "model_registry.json")
MDD_MODELS = {m["key"]: m for m in MDD["models"]}
assert {"fts_c", "nofts_c"} <= set(MDD_MODELS), "config.mdd.models must define fts_c and nofts_c"
assert os.path.abspath(MDD_MODELS["nofts_c"]["registry"]) == os.path.abspath(REG_PATH), "config.mdd nofts_c registry != this variant's registry"
assert os.path.abspath(MDD_MODELS["fts_c"]["registry"]) == os.path.abspath(FTS_REG_PATH), "config.mdd fts_c registry != compare_with_fts.registry"
DOC_ROOT = str(MDD["doc_output_root"])
OUT = os.path.join(DOC_ROOT, "reports", "benchmark")
MODEL_ROOTS = sorted({os.path.dirname(os.path.abspath(REG_PATH)), os.path.dirname(os.path.abspath(FTS_REG_PATH))})
os.makedirs(os.path.join(OUT, "run_summaries"), exist_ok=True)
XLSX_PATH = os.path.join(OUT, "benchmark_performance.xlsx")
FIG_ROC = os.path.join(OUT, "roc_common_sample.png")
FIG_CAP = os.path.join(OUT, "capture_common_sample.png")
SUMMARY_PATH = os.path.join(OUT, "run_summaries", "s05c_benchmark_performance.json")

cat, sch = cfg["databricks"]["catalog"], cfg["databricks"]["schema"]
MASTER_TBL = f"{cat}.{sch}.{cfg['tables']['master']}"
E13_STORED = f"{cat}.{sch}.{cfg['tables']['oot_predictions']}"   # no-FTS E13 case-level OOT scores
E13_STORED_FTS = f"{cat}.{sch}.acard_oot_predictions"             # FTS v1.0.1 E13 case-level OOT scores (same as s06b)
id_col, DATE_A, DECISION_A = cfg["columns"]["id"], cfg["columns"]["date"], cfg["columns"]["decision_time"]
WATCH = list(cfg["column_roles"]["watch"])

# Report-only constants (same values as s06b_cutoff_analysis; nothing is selected, dropped or fitted with them)
SEG_MIN_BADS = 50                      # Step 5: segments with < 50 bads -> bad rates only, no Gini
AGE_BANDS = [0, 25, 30, 35, 40, 45, 50, 60, 200]
TOP_N_WATCH = 10                       # other categories -> OTHER
AECB_LABEL = "reference only: pre-upgrade bureau score version in OOT, not a model input"

GIT_COMMIT = common.git_commit(PROJECT_ROOT)
CONFIG_HASH = common.config_hash(cfg)
RUN_TIME = datetime.now().isoformat(timespec="seconds")
WRITTEN = []
_t0 = time.time()

def _json_default(o):
    if hasattr(o, "item"):
        return o.item()
    if isinstance(o, float) and np.isnan(o):
        return None
    return str(o)

def _guard(dst):
    assert os.path.abspath(dst).startswith(os.path.abspath(OUT) + os.sep), f"refusing to write outside {OUT}: {dst}"

def _tmp(dst):
    return os.path.join(tempfile.mkdtemp(prefix="s05c_"), os.path.basename(dst))

def write_json(obj, dst):
    """Local temp file, then copy to the UC volume (copyfile: FUSE does not allow chmod/utime)."""
    _guard(dst)
    tmp = _tmp(dst)
    with open(tmp, "w") as fh:
        fh.write(json.dumps(obj, indent=2, default=_json_default))
    shutil.copyfile(tmp, dst)
    WRITTEN.append(dst)

def save_fig(fig, dst):
    _guard(dst)
    tmp = _tmp(dst)
    fig.savefig(tmp, dpi=130, bbox_inches="tight")
    plt.close(fig)
    shutil.copyfile(tmp, dst)
    WRITTEN.append(dst)

def save_excel(sheets, dst):
    """sheets: name -> (description, [(title, DataFrame), ...]). Row 1 of every sheet = its description."""
    _guard(dst)
    tmp = _tmp(dst)
    with pd.ExcelWriter(tmp, engine="openpyxl") as xw:
        for name, (desc, blocks) in sheets.items():
            assert len(name) <= 31, f"sheet name too long: {name}"
            pd.DataFrame(columns=[desc]).to_excel(xw, sheet_name=name, startrow=0, index=False)
            r = 2
            for title, df in blocks:
                df = df.copy()
                for c in df.columns:
                    if isinstance(df[c].dtype, pd.PeriodDtype) or str(df[c].dtype) in ("string", "category") or str(df[c].dtype).startswith("interval"):
                        df[c] = df[c].astype(str)
                    elif df[c].dtype == object:
                        df[c] = df[c].map(lambda v: json.dumps(v, default=str) if isinstance(v, (list, dict, tuple)) else v)
                if title:
                    pd.DataFrame(columns=[title]).to_excel(xw, sheet_name=name, startrow=r, index=False)
                    r += 1
                df.to_excel(xw, sheet_name=name, startrow=r, index=False)
                r += len(df) + 3
    shutil.copyfile(tmp, dst)
    WRITTEN.append(dst)

print(f"Output     : {OUT}\nWorkbook   : {XLSX_PATH}")
print(f"Registries : no-FTS {REG_PATH} | FTS {FTS_REG_PATH}")
print(f"Git commit : {GIT_COMMIT[:12]} | Config hash: {CONFIG_HASH[:12]} | bootstrap {BOOT} | periods {PERIODS}")


In [ ]:
NF = bu.load_registry(REG_PATH, "no-FTS", MODELS_USED)       # sign-off, every locked file hash, Platt file hash
FTSR = bu.load_registry(FTS_REG_PATH, "FTS", MODELS_USED)
assert NF["version"] == str(cfg["registry"]["version"]), f"no-FTS registry version {NF['version']} != config.registry.version"
assert not [f for f in NF["FEATS"] if f.lower().startswith("fts_")], "ABORT: the no-FTS models use an fts_ feature"
FDICT = pd.read_csv(os.path.join(PROJECT_ROOT, "config", "feature_dict.csv"))

SEL = {R["label"]: common.resolve_selected_model(R["reg"], verbose=False) for R in (NF, FTSR)}
E13 = {R["label"]: {m: R["reg"]["oot_evaluation"]["metrics"][m]["overall"] for m in MODELS_USED} for R in (NF, FTSR)}

# before-run state: every file under both model folders (this module's output folder excluded) and the read tables
SNAP0 = {r: bu.tree_snapshot(r, exclude_dirs=[OUT]) for r in MODEL_ROOTS}
PROTECTED = list(dict.fromkeys([MASTER_TBL, E13_STORED, E13_STORED_FTS] + [b["table"] for b in BENCHES if b.get("table")]))
PROT0 = {t: bu.table_version(spark, t) for t in PROTECTED if spark.catalog.tableExists(t)}
for R in (NF, FTSR):
    rg = R["reg"]
    print(f"{R['label']}: v{rg['version']} locked {rg['locked_at']} | {len(R['files'])} locked hashes OK | Platt OK ({R['platt_sha'][:12]}) | "
          f"selected {SEL[R['label']]['model']} ({SEL[R['label']]['status']}) | E13 OOT Gini A {E13[R['label']]['A']['gini']:.4f}, C {E13[R['label']]['C']['gini']:.4f}")
print(f"Snapshots: {[(r, len(s)) for r, s in SNAP0.items()]} | tables {PROT0}")


In [ ]:
_ML = bu.load_master(spark, MASTER_TBL, id_col, DATE_A, DECISION_A, WATCH, [], BENCHES, NF["FEATS"] + FTSR["FEATS"])
master, WATCH_COLS, STATUS_COUNTS = _ML["master"], _ML["WATCH_COLS"], _ML["STATUS_COUNTS"]
OOT = bu.scope(master, "OOT", cfg, REG_PATH, BASE_STATUSES, DATE_A, DECISION_A)      # common.load_window (OOT read lock)
bu.add_oot_split(OOT, SPLITS, DATE_A)
bu.score_e13(OOT, NF, cfg, FDICT, MODELS_USED)                 # pd_A, pd_C, score_A, score_C
bu.score_e13(OOT, FTSR, cfg, FDICT, MODELS_USED, prefix="FTS_")  # pd_FTS_A, pd_FTS_C, ...
_BA = bu.attach_benchmarks(spark, BENCHES, [("OOT", OOT)], LABELLED, BASE_STATUSES, DATE_A, MASTER_TBL, bu.ASCORE_OFFSETS)
BENCH_SRC, BENCH_COV, BENCH_DIR_SUMM = _BA["BENCH_SRC"], _BA["BENCH_COV"], _BA["BENCH_DIR_SUMM"]

def verify_stored(R, prefix, stored):
    """All OOT rows (unique CREDITCARDNO in both) against the stored E13 case-level scores of the same registry."""
    st = spark.table(stored).select("CREDITCARDNO", *[f"prob_{m}_cal" for m in MODELS_USED], *[f"score_{m}" for m in MODELS_USED]).toPandas()
    st["CREDITCARDNO"] = st["CREDITCARDNO"].astype(str)
    st = st[~st["CREDITCARDNO"].duplicated(keep=False)].set_index("CREDITCARDNO")
    card = OOT["CREDITCARDNO"].astype(str)
    sel = np.where((~card.duplicated(keep=False) & card.isin(st.index)).to_numpy())[0]
    s_ = st.loc[card.iloc[sel]]
    gb = OOT["sample_status"].isin(LABELLED).to_numpy()
    return [{"registry": R["label"], "version": R["version"], "model": m, "stored_table": stored, "oot_rows": len(OOT),
             "oot_good_bad_rows": int(gb.sum()), "rows_compared": len(sel), "good_bad_rows_compared": int(gb[sel].sum()),
             "max_abs_pd_diff": float(np.abs(OOT[f"pd_{prefix}{m}"].to_numpy()[sel] - s_[f"prob_{m}_cal"].to_numpy()).max()),
             "n_score_diff": int((OOT[f"score_{prefix}{m}"].to_numpy()[sel] != s_[f"score_{m}"].to_numpy()).sum())} for m in MODELS_USED]

VERIFY = pd.DataFrame(verify_stored(NF, "", E13_STORED) + verify_stored(FTSR, "FTS_", E13_STORED_FTS))

# A_SCORE: independent re-join of the benchmark table (Spark, not benchmark_utils) with the same 0/1 score-month rule
_ab = next(b for b in BENCHES if b["key"] == "A_SCORE")
_snap = spark.table(_ab["table"]).selectExpr("trim(cast(app_id as string)) as app_id", "cast(a_score as double) as a_score_chk", "score_date").toPandas()
_chk = OOT[["app_id", DATE_A, "score_A_SCORE"]].merge(_snap, on="app_id", how="left", validate="m:1")
_off = (pd.to_datetime(_chk["score_date"]).dt.year - pd.to_datetime(_chk[DATE_A]).dt.year) * 12 + \
       (pd.to_datetime(_chk["score_date"]).dt.month - pd.to_datetime(_chk[DATE_A]).dt.month)
_exp = np.where(_chk["a_score_chk"].notna() & _off.isin(bu.ASCORE_OFFSETS), _chk["a_score_chk"], np.nan)
ASCORE_RECHECK = {"rows": len(_chk), "used": int(np.isfinite(_exp).sum()),
                  "identical": bool(np.array_equal(_exp, _chk["score_A_SCORE"].to_numpy(), equal_nan=True))}
_aecb = next(b for b in BENCHES if b["key"] == "AECB_SCORE")
AECB_RECHECK = {"identical": bool(np.array_equal(pd.to_numeric(OOT[_aecb["_master_col"]], errors="coerce").to_numpy(dtype=float) *
                                                 (1.0 if _aecb["higher_is_lower_risk"] else -1.0), OOT["score_AECB_SCORE"].to_numpy(), equal_nan=True))}
print("OOT approval base:", len(OOT), OOT["sample_status"].value_counts().to_dict())
print("A_SCORE re-join:", ASCORE_RECHECK, "| AECB re-read:", AECB_RECHECK)
display(VERIFY)
display(BENCH_DIR_SUMM)


In [ ]:
def _bench_label(k):
    return f"{BENCH_NAME_OF[k]} ({AECB_LABEL})" if k == "AECB_SCORE" else f"{BENCH_NAME_OF[k]} ({bu.BENCH_DESC.get(k, '')})"

# key -> label, risk column (higher = riskier), native score column (higher = lower risk), role
SCORES = {
    "fts_c":      {"label": MDD_MODELS["fts_c"]["label"], "pd": "pd_FTS_C", "score": "score_FTS_C", "role": "candidate"},
    "nofts_c":    {"label": MDD_MODELS["nofts_c"]["label"], "pd": "pd_C", "score": "score_C", "role": "candidate"},
    "A_SCORE":    {"label": _bench_label("A_SCORE"), "pd": None, "score": "score_A_SCORE", "role": "benchmark"},
    "AECB_SCORE": {"label": _bench_label("AECB_SCORE"), "pd": None, "score": "score_AECB_SCORE", "role": "benchmark, reference only"},
    "fts_a":      {"label": f"Model A with FTS ({FTSR['version']}), reference", "pd": "pd_FTS_A", "score": "score_FTS_A", "role": "reference"},
    "nofts_a":    {"label": f"Model A without FTS ({NF['version']}), reference", "pd": "pd_A", "score": "score_A", "role": "reference"},
}
PRIMARY = ["fts_c", "nofts_c", "A_SCORE", "AECB_SCORE"]
PAIRS = [("fts_c", "A_SCORE"), ("nofts_c", "A_SCORE"), ("fts_c", "AECB_SCORE"), ("nofts_c", "AECB_SCORE"), ("fts_c", "nofts_c")]

GB = OOT[OOT["sample_status"].isin(LABELLED)].reset_index(drop=True)
Y = (GB["sample_status"] == "BAD").astype(int).to_numpy()
RISK, HAS = {}, {}
for k, s in SCORES.items():
    # models: calibrated PD (Gini = E13, no ties from score rounding); benchmarks: oriented score, negated (higher = riskier)
    RISK[k] = GB[s["pd"]].to_numpy(dtype=float) if s["pd"] else -GB[s["score"]].to_numpy(dtype=float)
    HAS[k] = np.isfinite(RISK[k])
assert all(HAS[k].all() for k in ("fts_c", "nofts_c", "fts_a", "nofts_a")), "a model score is missing for an OOT GOOD/BAD row"
COMMON = np.logical_and.reduce([HAS[k] for k in PRIMARY])
PMASK = {"ALL": np.ones(len(GB), bool), **{q: (GB["oot_split"] == q).to_numpy() for q in SPLITS}}

POP = []
for p in PERIODS:
    for nm, mk in (("OOT GOOD/BAD", PMASK[p]), ("COMMON (all four scores)", PMASK[p] & COMMON), ("excluded from COMMON", PMASK[p] & ~COMMON)):
        POP.append({"population": nm, "period": p, "n": int(mk.sum()), "bads": int(Y[mk].sum()), "goods": int((1 - Y[mk]).sum()),
                    "bad_rate": Y[mk].mean() if mk.any() else np.nan})
POP = pd.DataFrame(POP)
COVER = pd.DataFrame([{"score": k, "label": s["label"], "role": s["role"], "period": p, "good_bad_n": int(PMASK[p].sum()),
                       "available_n": int((HAS[k] & PMASK[p]).sum()), "available_bads": int(Y[HAS[k] & PMASK[p]].sum()),
                       "coverage": (HAS[k] & PMASK[p]).sum() / PMASK[p].sum(), "in_common_n": int((COMMON & PMASK[p]).sum())}
                      for k, s in SCORES.items() for p in PERIODS])
EXCL_WHY = pd.DataFrame([{"missing_score": k, "rows_missing": int((~HAS[k]).sum()), "bads_missing": int(Y[~HAS[k]].sum())} for k in PRIMARY])
print(f"OOT GOOD/BAD {len(GB)} ({Y.sum()} bads) | COMMON {COMMON.sum()} ({Y[COMMON].sum()} bads) | excluded {(~COMMON).sum()} ({Y[~COMMON].sum()} bads)")
display(POP)
display(EXCL_WHY)


In [ ]:
_CACHE = {}
def metrics(k, mk):
    """common.bootstrap_auc_gini_ks (E13 function, config.selection.bootstrap) on rows mk with score k; cached per row set."""
    idx = np.where(mk & HAS[k])[0]
    ck = (k, hashlib.sha1(idx.tobytes()).hexdigest())
    if ck not in _CACHE:
        _CACHE[ck] = common.bootstrap_auc_gini_ks(Y[idx], RISK[k][idx], cfg)
    return dict(_CACHE[ck])

def capture(k, mk, top):
    """Share of bads among the riskiest `top` share of rows (ranks with ties broken by row order)."""
    idx = np.where(mk & HAS[k])[0]
    r = pd.Series(-RISK[k][idx]).rank(method="first").to_numpy()
    return float(Y[idx][r <= np.floor(top * len(idx))].sum() / Y[idx].sum())

STEP1 = []
for view in ("COMMON", "OWN"):
    for p in PERIODS:
        for k, s in SCORES.items():
            mk = PMASK[p] & (COMMON if view == "COMMON" else HAS[k])
            STEP1.append({"view": view, "period": p, "score": k, "label": s["label"], "role": s["role"], **metrics(k, mk),
                          "top10_capture": capture(k, mk, 0.10), "top20_capture": capture(k, mk, 0.20)})
STEP1 = pd.DataFrame(STEP1)

# E13 reconciliation: full OOT GOOD/BAD Gini must equal the registry value; the COMMON value differs by the sample only
E13REC = []
for R, ks in ((NF, {"A": "nofts_a", "C": "nofts_c"}), (FTSR, {"A": "fts_a", "C": "fts_c"})):
    for m, k in ks.items():
        full, com = metrics(k, PMASK["ALL"]), metrics(k, COMMON)
        E13REC.append({"registry": R["label"], "version": R["version"], "model": m, "score": k,
                       "e13_registry_gini": E13[R["label"]][m]["gini"], "e13_registry_gini_lo": E13[R["label"]][m]["gini_lo"],
                       "e13_registry_gini_hi": E13[R["label"]][m]["gini_hi"], "full_gini_this_run": full["gini"],
                       "abs_diff_vs_e13": abs(full["gini"] - E13[R["label"]][m]["gini"]), "full_n": full["n"], "full_bads": full["n_bad"],
                       "common_gini": com["gini"], "common_n": com["n"], "common_bads": com["n_bad"],
                       "common_minus_full": com["gini"] - full["gini"],
                       "explanation": f"same case-level scores; COMMON drops {full['n'] - com['n']} rows ({full['n_bad'] - com['n_bad']} bads) "
                                      f"without a usable A_SCORE, so the difference is the sample only"})
E13REC = pd.DataFrame(E13REC)
display(STEP1[(STEP1["period"] == "ALL")][["view", "score", "n", "n_bad", "gini", "gini_lo", "gini_hi", "ks", "auc", "top10_capture", "top20_capture"]])
display(E13REC)


In [ ]:
def paired(x, yk, mk, period, extra=None):
    """common.paired_bootstrap (E13 function): delta = x - y, stratified resampling of the same rows for both scores."""
    idx = np.where(mk)[0]
    pb = common.paired_bootstrap(Y[idx], RISK[x][idx], RISK[yk][idx], cfg).set_index("metric")
    mx, my = metrics(x, mk), metrics(yk, mk)
    out = []
    for d, obs in (("dGini", mx["gini"] - my["gini"]), ("dAUC", mx["auc"] - my["auc"]), ("dKS", mx["ks"] - my["ks"])):
        out.append({**(extra or {}), "period": period, "x": x, "y": yk, "x_label": SCORES[x]["label"], "y_label": SCORES[yk]["label"],
                    "n": len(idx), "bads": int(Y[idx].sum()), "metric": d, "x_value": mx[d[1:].lower()], "y_value": my[d[1:].lower()],
                    "delta_observed": obs, "delta_boot_mean": pb.loc[d, "mean"], "ci_low": pb.loc[d, "ci_low"], "ci_high": pb.loc[d, "ci_high"],
                    "p_delta_le_0": pb.loc[d, "p_le_0"]})
    return out

STEP2 = pd.DataFrame([r for p in PERIODS for x, yk in PAIRS for r in paired(x, yk, PMASK[p] & COMMON, p)])
display(STEP2[STEP2["metric"] == "dGini"][["period", "x", "y", "n", "bads", "x_value", "y_value", "delta_observed", "ci_low", "ci_high", "p_delta_le_0"]])


In [ ]:
def deciles(k, mk):
    """Decile 1 = riskiest 10% by score k (ties broken by row order); cumulative bad capture from decile 1."""
    idx = np.where(mk & HAS[k])[0]
    d = pd.DataFrame({"y": Y[idx], "score": GB[SCORES[k]["score"]].to_numpy()[idx]})
    d["decile"] = pd.qcut(pd.Series(-RISK[k][idx]).rank(method="first"), 10, labels=False).to_numpy() + 1
    t = d.groupby("decile").agg(n=("y", "size"), bads=("y", "sum"), score_min=("score", "min"), score_max=("score", "max")).reset_index()
    t["goods"] = t["n"] - t["bads"]
    t["bad_rate"] = t["bads"] / t["n"]
    t["cum_population_share"] = t["n"].cumsum() / t["n"].sum()
    t["cum_bad_capture"] = t["bads"].cumsum() / t["bads"].sum()
    t["lift"] = t["bad_rate"] / (t["bads"].sum() / t["n"].sum())
    return t, pd.Series(d["decile"].to_numpy(), index=idx)

DEC, DEC_IDX = [], {}
for view in ("COMMON", "OWN"):
    for k, s in SCORES.items():
        t, di = deciles(k, COMMON if view == "COMMON" else HAS[k])
        DEC.append(t.assign(view=view, score=k, label=s["label"]))
        if view == "COMMON":
            DEC_IDX[k] = di
DEC = pd.concat(DEC, ignore_index=True)[["view", "score", "label", "decile", "n", "goods", "bads", "bad_rate", "score_min", "score_max",
                                         "cum_population_share", "cum_bad_capture", "lift"]]
CAPT = STEP1[STEP1["period"] == "ALL"][["view", "score", "label", "n", "n_bad", "top10_capture", "top20_capture"]].reset_index(drop=True)

# Step 4: Spearman rank correlation of the risk orientation (higher = riskier) on the common sample
KEYS = list(SCORES)
ci = np.where(COMMON)[0]
RHO = pd.DataFrame([[spearmanr(RISK[a][ci], RISK[b][ci]).correlation for b in KEYS] for a in KEYS], index=KEYS, columns=KEYS)
RHO = RHO.reset_index(names="score")
XTAB = []
for x in ("fts_c", "nofts_c"):
    for b in ("A_SCORE", "AECB_SCORE"):
        dx, db = DEC_IDX[x].loc[ci].to_numpy(), DEC_IDX[b].loc[ci].to_numpy()
        cnt = pd.crosstab(pd.Series(dx, name=f"{x}_decile"), pd.Series(db, name=f"{b}_decile"))
        brt = pd.crosstab(pd.Series(dx, name=f"{x}_decile"), pd.Series(db, name=f"{b}_decile"), values=Y[ci], aggfunc="mean")
        same = float((dx == db).mean())
        within1 = float((np.abs(dx - db) <= 1).mean())
        XTAB += [(f"Counts: {x} decile (rows) vs {SCORES[b]['label']} decile (columns); decile 1 = riskiest; same decile {same:.1%}, "
                  f"within one decile {within1:.1%}", cnt.reset_index()),
                 (f"Bad rate: {x} decile (rows) vs {b} decile (columns)", brt.reset_index())]
display(CAPT[CAPT["view"] == "COMMON"])
display(RHO)


In [ ]:
def watch_frame(df):
    """Same banding as s06b_cutoff_analysis Step 4.4 (computed on the OOT approval base), plus the has_fts_data flag."""
    out = {"has_fts_data": df["has_fts_data"].astype(str)}
    for w in WATCH_COLS:
        if w.upper() == "AGE":
            b = pd.cut(pd.to_numeric(df[w], errors="coerce"), AGE_BANDS, right=False)
            out["AGE_BAND"] = b.astype(object).where(b.notna(), "MISSING").astype(str)
        else:
            v = common._norm(df[w])
            v = v.astype(object).where(v.notna(), "MISSING").astype(str)
            top = v.value_counts().index[:TOP_N_WATCH]
            out[w] = v.where(v.isin(top), "OTHER")
    return pd.DataFrame(out, index=df.index)

WF = watch_frame(OOT)
WF = WF[OOT["sample_status"].isin(LABELLED).to_numpy()].reset_index(drop=True)      # same row order as GB
assert len(WF) == len(GB)
SEG, SEGD = [], []
for dim in WF.columns:
    for val in sorted(WF[dim].unique()):
        mk = (WF[dim] == val).to_numpy() & COMMON
        if not mk.any():
            continue
        nb = int(Y[mk].sum())
        with_gini = nb >= SEG_MIN_BADS
        base = {"dim": dim, "value": val, "n": int(mk.sum()), "bads": nb, "bad_rate": Y[mk].mean(),
                "note": "" if with_gini else f"< {SEG_MIN_BADS} bads: bad rate only, no Gini"}
        if not with_gini:
            SEG.append(base)
            continue
        for k, s in SCORES.items():
            m = metrics(k, mk)
            SEG.append({**base, "score": k, "label": s["label"], **{c: m[c] for c in ("auc", "gini", "gini_lo", "gini_hi", "ks", "ks_lo", "ks_hi")}})
        for x, yk in PAIRS:
            SEGD += [r for r in paired(x, yk, mk, "ALL", extra={"dim": dim, "value": val}) if r["metric"] == "dGini"]
SEG, SEGD = pd.DataFrame(SEG), pd.DataFrame(SEGD)
display(SEG[SEG["score"].isin(["fts_c", "nofts_c", "A_SCORE"]) | SEG["score"].isna()][["dim", "value", "n", "bads", "score", "gini", "gini_lo", "gini_hi", "note"]])


In [ ]:
FPR_GRID = np.round(np.linspace(0, 1, 101), 2)
ROC, CAP = [], []
for k, s in SCORES.items():
    fpr, tpr, _ = roc_curve(Y[ci], RISK[k][ci])
    ROC += [{"score": k, "label": s["label"], "fpr": f, "tpr": float(np.interp(f, fpr, tpr))} for f in FPR_GRID]
    r = pd.Series(-RISK[k][ci]).rank(method="first").to_numpy()
    yb = Y[ci][np.argsort(r)]
    cb = np.concatenate([[0], np.cumsum(yb)]) / yb.sum()
    CAP += [{"score": k, "label": s["label"], "share_declined_riskiest": f, "bad_capture": float(cb[int(np.floor(f * len(yb)))])} for f in FPR_GRID]
ROC, CAP = pd.DataFrame(ROC), pd.DataFrame(CAP)

_g = STEP1[(STEP1["view"] == "COMMON") & (STEP1["period"] == "ALL")].set_index("score")["gini"]
STYLE = {"fts_c": ("tab:blue", "-"), "nofts_c": ("tab:orange", "-"), "A_SCORE": ("tab:green", "-"), "AECB_SCORE": ("tab:gray", "--"),
         "fts_a": ("tab:blue", ":"), "nofts_a": ("tab:orange", ":")}
for tbl, xc, yc, ttl, xl, yl, dst in ((ROC, "fpr", "tpr", "ROC", "False positive rate (goods declined)", "True positive rate (bads declined)", FIG_ROC),
                                      (CAP, "share_declined_riskiest", "bad_capture", "Cumulative bad capture", "Share of applicants declined (riskiest first)",
                                       "Share of bads captured", FIG_CAP)):
    fig, ax = plt.subplots(figsize=(7, 6))
    for k in SCORES:
        d = tbl[tbl["score"] == k]
        c, ls = STYLE[k]
        ax.plot(d[xc], d[yc], color=c, linestyle=ls, label=f"{k}: Gini {_g[k]:.3f}" + (" (reference only)" if k == "AECB_SCORE" else ""))
    ax.plot([0, 1], [0, 1], color="black", lw=0.6)
    ax.set_title(f"{ttl}, OOT GOOD/BAD common sample (n={len(ci)}, bads={int(Y[ci].sum())})")
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.legend(loc="lower right", fontsize=8); ax.grid(alpha=0.3)
    save_fig(fig, dst)
FIGS = pd.DataFrame([{"figure": os.path.basename(FIG_ROC), "path": FIG_ROC, "source_sheet": "step6_roc_curve"},
                     {"figure": os.path.basename(FIG_CAP), "path": FIG_CAP, "source_sheet": "step6_capture_curve"}])
print("Figures:", FIGS["path"].tolist())


In [ ]:
META = pd.DataFrame([("module", "s05c_benchmark_performance"), ("run_time", RUN_TIME), ("git_commit", GIT_COMMIT), ("config_sha256", CONFIG_HASH),
                     ("population", "OOT (config.windows) applicants with sample_status in config.labelled_statuses (GOOD, BAD)"),
                     ("primary_view", "COMMON sample: fts_c, nofts_c, A_SCORE and AECB_SCORE all non-missing"),
                     ("secondary_view", "OWN sample: each score on the rows where it is available"),
                     ("model_risk_input", "calibrated PD (Platt, DEV OOF) of the locked models, E13 scoring code (benchmark_utils.score_e13)"),
                     ("benchmark_risk_input", "benchmark score oriented so that higher = lower risk, negated for AUC (benchmark_utils.attach_benchmarks)"),
                     ("AECB_SCORE", AECB_LABEL),
                     ("bootstrap", json.dumps(BOOT)), ("metrics_functions", "common.bootstrap_auc_gini_ks, common.paired_bootstrap (same as E13)"),
                     ("segment_rule", f"segments with < {SEG_MIN_BADS} bads: bad rates only, no Gini"),
                     ("decile_rule", "decile 1 = riskiest 10%, ties broken by row order")], columns=["key", "value"])
REGS = pd.DataFrame([{"registry": R["label"], "path": R["path"], "version": R["version"], "registry_sha256": R["reg_sha"], "locked_at": R["reg"]["locked_at"],
                      "locked_files": len(R["files"]), "platt_sha256": R["platt_sha"], "selected_model": SEL[R["label"]]["model"],
                      "selection_source": SEL[R["label"]]["source"], "selection_status": SEL[R["label"]]["status"],
                      "rule_selected_model": SEL[R["label"]].get("rule_selected_model")} for R in (NF, FTSR)])
SCORE_DEF = pd.DataFrame([{"score": k, **s} for k, s in SCORES.items()])
SHEETS = {
    "meta": ("Run metadata, registries, models, benchmark sources and definitions",
             [("Run metadata", META), ("Locked registries (read-only)", REGS), ("Scores compared", SCORE_DEF),
              ("Benchmark sources (config.cutoff_analysis.compare_with)", BENCH_SRC), ("Benchmark direction check (OOT GOOD/BAD with the benchmark)", BENCH_DIR_SUMM),
              ("Scores vs stored E13 case-level scores (all OOT rows with a unique CREDITCARDNO)", VERIFY)]),
    "populations": ("OOT GOOD/BAD populations and the COMMON sample", [("Populations", POP), ("Rows missing each primary score", EXCL_WHY)]),
    "coverage": ("Coverage of each score on OOT GOOD/BAD", [(None, COVER), ("Benchmark coverage on the OOT approval base", BENCH_COV)]),
    "step1_discrimination": ("Step 1: AUC, Gini, KS with stratified bootstrap 95% CIs; COMMON (primary) and OWN (secondary) sample; overall and by OOT quarter",
                             [(None, STEP1)]),
    "e13_reconciliation": ("Full OOT GOOD/BAD Gini vs the E13 registry value, and the COMMON-sample value", [(None, E13REC)]),
    "step2_paired_deltas": ("Step 2: paired stratified bootstrap differences (x - y) on the COMMON sample; p_delta_le_0 = P(delta <= 0)", [(None, STEP2)]),
    "step3_deciles": ("Step 3: bad rate by score decile (decile 1 = riskiest), cumulative bad capture", [(None, DEC)]),
    "step3_capture": ("Step 3: bad capture in the riskiest 10% and 20%", [(None, CAPT)]),
    "step4_rank_corr": ("Step 4: Spearman rank correlation (risk orientation) on the COMMON sample", [(None, RHO)]),
    "step4_cross_tables": ("Step 4: decile cross-tables, Model C vs each benchmark, COMMON sample", XTAB),
    "step5_segments": (f"Step 5: Step 1 by segment on the COMMON sample (< {SEG_MIN_BADS} bads: bad rate only)", [(None, SEG)]),
    "step5_segment_deltas": (f"Step 5: Step 2 (dGini) by segment on the COMMON sample (segments with >= {SEG_MIN_BADS} bads)", [(None, SEGD)]),
    "step6_roc_curve": ("Step 6: ROC curve points (COMMON sample); figure roc_common_sample.png is drawn from this table", [(None, ROC)]),
    "step6_capture_curve": ("Step 6: cumulative bad capture curve (COMMON sample); figure capture_common_sample.png is drawn from this table", [(None, CAP)]),
    "figures": ("Figure files in the output folder", [(None, FIGS)]),
}
save_excel(SHEETS, XLSX_PATH)

import mlflow
mlflow.set_experiment(cfg["databricks"]["mlflow_experiment"])
with mlflow.start_run(run_name="s05c_benchmark_performance") as run:
    mlflow.log_params({"module": "s05c_benchmark_performance", "config_sha256": CONFIG_HASH, "git_commit": GIT_COMMIT,
                       "nofts_registry_sha256": NF["reg_sha"], "fts_registry_sha256": FTSR["reg_sha"], "common_n": int(COMMON.sum()),
                       "common_bads": int(Y[COMMON].sum()), "bootstrap_n": BOOT["n"], "bootstrap_seed": BOOT["seed"]})
    for _, r in STEP1[(STEP1["view"] == "COMMON")].iterrows():
        mlflow.log_metric(f"gini_common_{r['period']}_{r['score']}", r["gini"])
    for _, r in STEP2[STEP2["metric"] == "dGini"].iterrows():
        mlflow.log_metric(f"dgini_{r['period']}_{r['x']}_vs_{r['y']}", r["delta_observed"])
        mlflow.log_metric(f"p_le0_{r['period']}_{r['x']}_vs_{r['y']}", r["p_delta_le_0"])
    for f in (XLSX_PATH, FIG_ROC, FIG_CAP):
        mlflow.log_artifact(f)
    MLFLOW_RUN_ID = run.info.run_id
print(f"Workbook: {XLSX_PATH} ({len(SHEETS)} sheets) | MLflow run {MLFLOW_RUN_ID} | {time.time() - _t0:.0f}s")


In [ ]:
import openpyxl
CHECKS = {}
# scores: same E13 code as s06b (benchmark_utils), reproduce the stored E13 case-level scores of both registries on every compared OOT row
for _, r in VERIFY.iterrows():
    CHECKS[f"{r['registry']} {r['model']}: scores = stored E13 ({r['rows_compared']} OOT rows, max |dPD| {r['max_abs_pd_diff']:.1e})"] = \
        bool(r["n_score_diff"] == 0 and r["max_abs_pd_diff"] <= TOL and r["good_bad_rows_compared"] == r["oot_good_bad_rows"])
CHECKS["A_SCORE = independent re-join of the benchmark table (0/1 score-month rule)"] = ASCORE_RECHECK["identical"]
CHECKS["AECB_SCORE = acard_master column, oriented"] = AECB_RECHECK["identical"]
# Model C (and A) full-sample OOT Gini = E13 registry value; COMMON-sample value reported separately
for _, r in E13REC.iterrows():
    CHECKS[f"{r['registry']} {r['model']}: full OOT GOOD/BAD Gini {r['full_gini_this_run']:.6f} = E13 {r['e13_registry_gini']:.6f}"] = bool(r["abs_diff_vs_e13"] <= TOL)
CHECKS["COMMON sample = OOT GOOD/BAD rows with all four primary scores non-missing"] = bool(
    COMMON.sum() == np.logical_and.reduce([np.isfinite(RISK[k]) for k in PRIMARY]).sum() and set(GB["sample_status"]) <= set(LABELLED))
CHECKS["Step 1 COMMON rows all use the same n"] = bool(STEP1[STEP1["view"] == "COMMON"].groupby("period")["n"].nunique().eq(1).all())
# registries, locked files, Platt files, model folders, tables: unchanged
for R in (NF, FTSR):
    CHECKS[f"{R['label']} registry file unchanged"] = bu.sha256_file(R["path"]) == R["reg_sha"]
    CHECKS[f"{R['label']}: all {len(R['files'])} locked file hashes unchanged"] = all(
        bu.sha256_file(v["path"]) == v["sha256"] for v in bu.read_json(R["path"])["files"].values())
    CHECKS[f"{R['label']}: Platt file unchanged"] = bu.sha256_file(R["platt_path"]) == R["platt_sha"] == R["cal_ref"]["sha256"]
for r, s0 in SNAP0.items():
    CHECKS[f"no file created/changed under {r} outside reports/benchmark"] = bu.tree_snapshot(r, exclude_dirs=[OUT]) == s0
CHECKS["read tables unchanged (Delta versions)"] = {t: bu.table_version(spark, t) for t in PROT0} == PROT0
CHECKS["every file written is under reports/benchmark"] = all(os.path.abspath(p).startswith(os.path.abspath(OUT) + os.sep) for p in WRITTEN)
# workbook
_wb = openpyxl.load_workbook(XLSX_PATH, read_only=True)
CHECKS["workbook: first sheet is meta, all analysis sheets present"] = _wb.sheetnames[0] == "meta" and _wb.sheetnames == list(SHEETS)
CHECKS["AECB_SCORE labelled reference only wherever it appears"] = bool(
    all("reference only" in str(l) for t in (STEP1, DEC, CAPT, SEG.dropna(subset=["score"]) if "score" in SEG else SEG)
        for l in t.loc[t["score"] == "AECB_SCORE", "label"]) and (STEP2.loc[STEP2["y"] == "AECB_SCORE", "y_label"].str.contains("reference only")).all())
CHECKS["figures written"] = all(os.path.exists(p) for p in (FIG_ROC, FIG_CAP))
CHECKS["bootstrap settings = config.selection.bootstrap"] = BOOT == cfg["selection"]["bootstrap"]

failed = [k for k, v in CHECKS.items() if not v]
summary = {"module": "s05c_benchmark_performance", "run_time": RUN_TIME, "git_commit": GIT_COMMIT, "config_sha256": CONFIG_HASH,
           "registries": REGS.to_dict("records"), "workbook": XLSX_PATH, "mlflow_run_id": MLFLOW_RUN_ID,
           "row_counts": {"oot_approval_base": len(OOT), "oot_good_bad": len(GB), "oot_bads": int(Y.sum()),
                          "common": int(COMMON.sum()), "common_bads": int(Y[COMMON].sum())},
           "key_metrics": {"gini_common": STEP1[(STEP1["view"] == "COMMON") & (STEP1["period"] == "ALL")].set_index("score")["gini"].to_dict(),
                           "dgini_common": {f"{r['x']}_vs_{r['y']}": {"delta": r["delta_observed"], "ci": [r["ci_low"], r["ci_high"]], "p_le_0": r["p_delta_le_0"]}
                                            for _, r in STEP2[(STEP2["metric"] == "dGini") & (STEP2["period"] == "ALL")].iterrows()}},
           "checks": CHECKS, "pass": not failed, "runtime_seconds": round(time.time() - _t0, 1)}
write_json(summary, SUMMARY_PATH)
for k, v in CHECKS.items():
    print("PASS" if v else "FAIL", k)
assert not failed, f"{len(failed)} acceptance check(s) failed: {failed}"
print(f"All {len(CHECKS)} acceptance checks PASS | run summary {SUMMARY_PATH}")
